# State_management

In [33]:
from typing import TypedDict
from langgraph.graph import StateGraph , START, END

class badstate(TypedDict):
    steps : list


def first_step(state : badstate) -> dict:
    current  =  state['steps']
    new_step = current + ['step_one']

    print(f'curent step is {new_step}')

    return{ 'steps':new_step}

In [34]:
def second_step(state:badstate) -> dict:
    current  =  state['steps']
    new_step = current + ['step_two']
    print(f'curent step is {new_step}')

    return{ 'steps':new_step}
    

In [35]:

builder = StateGraph(badstate)

builder.add_node('first_step',first_step)
builder.add_node('second_step',second_step)

builder.add_edge(START,'first_step')
builder.add_edge('first_step','second_step')
builder.add_edge('second_step',END)

In [36]:
graph = builder.compile()
result = graph.invoke({"steps": []})
print("Final step:", result["steps"])

curent step is ['step_one']
curent step is ['step_one', 'step_two']
Final step: ['step_one', 'step_two']


In [37]:
from typing import TypedDict,Annotated
from langgraph.graph import StateGraph , START, END
from operator import add

class GoodListState(TypedDict):
    steps : Annotated[list,add]



In [38]:
def first_step(state: GoodListState) -> dict:
    print(f"  step_one sees steps={state['steps']}")
    # Return ONLY the new item(s) - NOT the whole list
    return {"steps": ["first_step"]}


def second_step(state: GoodListState) -> dict:
    print(f"  step_two sees steps={state['steps']}")
    # Return ONLY the new item(s)
    return {"steps": ["second_step"]}


In [39]:

builder = StateGraph(GoodListState)

builder.add_node('first_step',first_step)
builder.add_node('second_step',second_step)

builder.add_edge(START,'first_step')
builder.add_edge('first_step','second_step')
builder.add_edge('second_step',END)

In [40]:
graph = builder.compile()
result = graph.invoke({"steps": []})
print("Final step:", result["steps"])

  step_one sees steps=[]
  step_two sees steps=['first_step']
Final step: ['first_step', 'second_step']


In [1]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

class chatstate(TypedDict):
    message : Annotated[list , add_messages]
    turn : int


In [2]:
def human_ques(state : chatstate) -> dict:
    new_mess = HumanMessage(content='what is capital of india')
    print(f'[human_input],appending :{new_mess.content}')

    return{
        'message':[new_mess],
        'turn':state['turn']+1
        
    }


def ai_reply(state : chatstate) -> dict:
    new_mess =  AIMessage(content='the capital of india is newdelhi')
    print(f'[ai_reply],appending{new_mess.content}')

    return{
        'message':[new_mess]
    }




In [8]:
builder = StateGraph(chatstate)

In [9]:
builder.add_node('human_ques',human_ques)
builder.add_node('ai_reply',ai_reply)

builder.add_edge(START,'human_ques')
builder.add_edge('human_ques','ai_reply')
builder.add_edge('ai_reply',END)

graph = builder.compile()

In [ ]:
result = graph.invoke({"message": [], "turn": 0})
print("\nFinal turn count:", result["turn"])
print("Final message list:")
for msg in result["message"]:
    role = "Human" if isinstance(msg, HumanMessage) else "AI"
    print(f"  [{role}] {msg.content}")
     

[human_input],appending :what is capital of india
[ai_reply],appendingthe capital of india is newdelhi

Final turn count: 1
Final message list:
  [Human] what is capital of india
  [AI] the capital of india is newdelhi


: 